samples effects to generate figures (for paper)

In [ ]:
import glow
import nibabel as nib
import numpy as np

# load experiment
num_img = 3
folder = '/home/matt/Dropbox/pnl_hglm/data/hcp100_lowres/image'
exp = glow.experiment.ExperimentImageOnly.from_search(folder=folder,
                                                      sbj_regex=r'[\d]{6}',
                                                      img_glob_dict={'FA': '*_FA.nii.gz'})
# truncate to first num_img images
exp.y = exp.y[:, :num_img, :]

In [ ]:
# set x value of each image according to its index
b, num_img, num_vox = exp.y.shape
exp = glow.experiment.Experiment(x=np.arange(num_img).reshape(1, num_img),
                                 contrast=np.array(True),
                                 add_bias=True,
                                 y=exp.y,
                                 mask_idx=exp.mask_idx)

# constrain ourselves to a slice (easier to visualize)
mask = exp.mask_idx > -1
mid = tuple(_xyz // 2 for _xyz in mask.shape)
mask[..., :mid[2]] = False
mask[..., mid[2] + 1:] = False
exp_slice = exp.apply_mask(mask)
exp_slice.mask_idx = exp_slice.mask_idx[..., mid[2]]

In [ ]:
def trim_nd_array(arr, trim_value=-1):
    """ (chatgpt)
    Trims all leading/trailing slices along every axis where all values == trim_value.
    """
    mask = arr != trim_value
    if not np.any(mask):
        # All elements are trim_value
        return np.empty((0,) * arr.ndim, dtype=arr.dtype)

    # Get bounding box
    coords = np.array(np.nonzero(mask))
    min_idx = coords.min(axis=1)
    max_idx = coords.max(axis=1) + 1

    # Build slicing tuple
    slices = tuple(slice(start, end) for start, end in zip(min_idx, max_idx))
    return arr[slices]

In [ ]:
exp_slice.mask_idx = trim_nd_array(exp_slice.mask_idx, trim_value=-1)

In [ ]:
import matplotlib.pyplot as plt
from skimage import measure

def plot_slice(y, mask_idx, mask=None, **kwargs):
    # nans transparent
    cmap = plt.cm.viridis.copy()
    cmap.set_bad(color=(0, 0, 0, 0)) 
    
    # build image
    img = np.full(mask_idx.shape, np.nan)
    img[mask_idx > -1] = y
    im = plt.imshow(img, cmap=cmap, **kwargs)
    
    ax = plt.gca()
    ax.grid(False)
    ax.tick_params(labelbottom=False, labelleft=False)
    ax.set_facecolor('none')
    
    if mask is not None:
        contours = measure.find_contours(mask, level=0.5)
        for contour in contours:
            ax.plot(contour[:, 1], contour[:, 0], linewidth=3, color='red')
    return im
    
def plot_all(exp, ax, mask=None, scatter=False, scatter_n=None, htitle=None, vminmax=None, num_img=3, num_img_scatter=3, title_flag=True,
             **kwargs):
    fig = plt.gcf()
    assert ax.size == num_img + scatter
    
    if vminmax is None:
        vmin, vmax = exp.y[0, ...].min(), exp.y[0, ...].max()
    else:
        vmin, vmax = vminmax
    
    img_idx = np.linspace(0, exp.y.shape[1] - 1, num=num_img, dtype=int)
    for _img_idx, _ax in zip(img_idx, ax):
        plt.sca(_ax)
        im = plot_slice(y=exp.y[0, _img_idx, :], 
                        mask_idx=exp_slice.mask_idx,
                        vmin=vmin, vmax=vmax, mask=mask,
                        **kwargs)
        if title_flag:
            _ax.set_title(f'img{_img_idx} (x={exp_slice.x[1, _img_idx]:g})')
        
    if scatter:
        # adds scatter plot of features
        y = exp.y[0, :, :]
        if mask is not None:
            y = y[:, exp.mask_idx[mask]]
            
        # get representative subset (less overwhelming visuals)
        num_vox = y.shape[1]
        if scatter_n is not None and scatter_n < num_vox:
            idx = np.linspace(0, num_vox - 1, scatter_n).astype(int)
            y = np.vstack([np.sort(row)[idx] for row in y])
            
        # plot with last x feature (avoids bias term)
        _x = exp.x[-1, :]
        x = np.broadcast_to(_x[:, None], y.shape)
        y_hat = y.mean(axis=1) @ np.linalg.pinv(exp.x) @ exp.x
        
        # trim to num_img_scatter
        img_idx = np.linspace(0, exp.y.shape[1] - 1, num=num_img_scatter, dtype=int)
        x = x[img_idx, :]
        y = y[img_idx, :]
        y_hat = y_hat[img_idx]
        
        # plot
        plt.sca(ax[-1])
        plt.plot(x.mean(axis=1), y_hat, color='k', linewidth=.5)
        plt.scatter(x=x.ravel(), y=y.flatten(), marker='s', s=13, c=y.flatten(),
                    vmin=vmin, vmax=vmax, cmap=im.get_cmap(), edgecolors='black', linewidths=0.4)
        
        labels = [f'x={val:g}\nimg{val:g}' for idx, val in enumerate(_x[img_idx])]
        ax[-1].set_xticks(_x[img_idx])
        ax[-1].set_xticklabels(labels)
        ax[-1].set_ylim(vmin, vmax)

    ax[0].set_ylabel(htitle)
    
    fig.tight_layout()

In [ ]:
# sample extent
ext_sphere = glow.effect.ExtenterSphere(radius=20)
mask = ext_sphere(y=exp_slice.y, mask_idx=exp_slice.mask_idx, seed=0)

# build effects
exp_tit_list = []
hotel_tr_all = [0, 1]
for hotel_tr in hotel_tr_all:
    _exp, effect = exp_slice.impose_effect(hotel_tr=hotel_tr, mask=mask, seed=0) 
    title = f'Hotelling Tr={effect.hotel_tr}'
    exp_tit_list.append((_exp, title))

fig, ax = plt.subplots(len(hotel_tr_all), num_img + 1)
    
# plot
vmin = min(exp.y.min() for exp, _ in exp_tit_list)
vmax = min(exp.y.max() for exp, _ in exp_tit_list)
vmin, vmax = -.2, 1.2
for idx, (_exp, title) in enumerate(exp_tit_list):
    plot_all(_exp, ax = ax[idx, :], mask=mask, scatter=True, scatter_n=20, htitle=title, vminmax=(vmin, vmax),
            title_flag=not idx)
    
fig.set_size_inches((12, 5))
plt.subplots_adjust(
    left=0.01,   # space from left edge of figure
    right=0.99,  # space from right edge
    top=0.99,    # space from top edge
    bottom=0.01, # space from bottom
    wspace=0.3, # horizontal space between subplots
    hspace=0.4  # vertical space between subplots
)
plt.savefig('effect_vary_hotel_tr.pdf', format='pdf', bbox_inches='tight')

In [ ]:
# sample extent
ext_sphere = glow.effect.ExtenterSphere(radius=20)
mask = ext_sphere(y=exp_slice.y, mask_idx=exp_slice.mask_idx, seed=0)

# build effects
exp_tit_list = []
rough = [.1, 1]
for _rough in rough:
    _exp, effect = exp_slice.impose_effect(hotel_tr=1, rough=_rough, mask=mask, seed=0) 
    title = r'$\lambda$=' + str(_rough)
    exp_tit_list.append((_exp, title))

fig, ax = plt.subplots(len(rough), num_img + 1)
    
# plot
vmin = min(exp.y.min() for exp, _ in exp_tit_list)
vmax = min(exp.y.max() for exp, _ in exp_tit_list)
vmin, vmax = -.2, 1
for idx, (_exp, title) in enumerate(exp_tit_list):
    plot_all(_exp, ax = ax[idx, :], mask=mask, scatter=True, scatter_n=20, htitle=title, vminmax=(vmin, vmax),
            title_flag=not idx)
    
fig.set_size_inches((12, 5))
plt.subplots_adjust(
    left=0.01,   # space from left edge of figure
    right=0.99,  # space from right edge
    top=0.99,    # space from top edge
    bottom=0.01, # space from bottom
    wspace=0.3, # horizontal space between subplots
    hspace=0.4  # vertical space between subplots
)
plt.savefig('effect_vary_rough.pdf', format='pdf', bbox_inches='tight')

In [ ]:
# sample extent
shape = 1, 5
mask_list = list()
ext_minvar = glow.effect.ExtenterMinVar(n=400)
for seed in range(np.prod(shape)):
    mask_list.append(ext_minvar(y=exp_slice.y, mask_idx=exp_slice.mask_idx, seed=seed))

fig, ax = plt.subplots(*shape)
ax = ax.reshape(shape)

y = exp_slice.y[0, :, :].mean(axis=0)
for idx, _mask in enumerate(mask_list):
    i, j = np.unravel_index(idx, shape)
    plt.sca(ax[i, j])
    plot_slice(y, mask_idx=exp_slice.mask_idx, mask=_mask, vmin=-0.2, vmax=1)
    
fig.set_size_inches((15, 5))
plt.subplots_adjust(
    wspace=0.3, # horizontal space between subplots
    hspace=0.4  # vertical space between subplots
)
plt.savefig('mask.pdf', format='pdf', bbox_inches='tight')